# Mini Project 3: Fashion Item Recognition

**Goal:** build a classifier that recognises clothing categories (T-shirts, trousers, pullovers, dresses, coats, sandals, shirts,
sneakers, bags and ankle boots) from 28 x 28 grayscale images, using the **Fashion-MNIST** dataset.

| Required output | Where it is in this notebook |
|---|---|
| Working notebook | This file, from top to bottom |
| Comparison table | Step 5 (three models side by side) |
| Prediction gallery | Step 8 (random gallery and one gallery per category) |
| Saved model | Step 10 (`fashion_cnn.pth`, reloaded and checked) |

**The plan.** We train **three models** on the same data and compare them:
1. a **fully connected baseline** (no convolution),
2. a **small CNN** (the one from Assignment 3),
3. the **final CNN**: more filters, **batch normalisation**, **dropout** and **data augmentation** (random flips and shifts of the training images).

The best model by validation accuracy becomes the saved model, and the test set is used only once, at the end.

**Runtime:** about 10 to 15 minutes on a laptop CPU. To check that everything works first, set `QUICK_RUN = True` in Step 1 (about 3 minutes).

## Step 1: Imports and settings

In [ ]:
import copy
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

QUICK_RUN = False          # True = very short training, only to test that the notebook runs

if QUICK_RUN:
    EPOCHS_MLP, EPOCHS_SMALL, EPOCHS_FINAL = 2, 2, 3
else:
    EPOCHS_MLP, EPOCHS_SMALL, EPOCHS_FINAL = 10, 8, 10
LEARNING_RATE = 0.001
BATCH_SIZE = 128

torch.manual_seed(42)
np.random.seed(42)
print("PyTorch version:", torch.__version__)
print("Epochs: fully connected", EPOCHS_MLP, "| small CNN", EPOCHS_SMALL, "| final CNN", EPOCHS_FINAL)

## Step 2: Load and prepare the data

- The first run downloads Fashion-MNIST (about 30 MB) into `fashion_mnist_data` in your user folder.
- Pixels are **normalised** with the training-set mean and standard deviation.
- 54,000 training images and 6,000 validation images are chosen with a fixed random split.
- **Augmentation** (random horizontal flips and small shifts) is applied only to the training images of the final model.
  Validation and test images are never changed, so the scores stay honest.

In [ ]:
DATA_DIR = os.path.join(os.path.expanduser("~"), "fashion_mnist_data")
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

raw_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transforms.ToTensor())
test_raw = datasets.FashionMNIST(DATA_DIR, train=False, download=True).data.numpy()   # original test pixels, for galleries

pixels = raw_train.data.float() / 255.0
MEAN, STD = pixels.mean().item(), pixels.std().item()
print(f"Training-set mean {MEAN:.4f} | standard deviation {STD:.4f}")

plain_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))])
augment_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(28, padding=2),          # shifts the item by up to 2 pixels
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

train_plain = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=plain_tf)
train_aug = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=augment_tf)
test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=plain_tf)

order = torch.randperm(60000, generator=torch.Generator().manual_seed(42)).tolist()
train_idx, val_idx = order[:54000], order[54000:]

train_loader_plain = DataLoader(Subset(train_plain, train_idx), batch_size=BATCH_SIZE, shuffle=True)
train_loader_aug = DataLoader(Subset(train_aug, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Subset(train_plain, val_idx), batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)       # not shuffled, so row i matches test_raw[i]

print("Train:", len(train_idx), "| Validation:", len(val_idx), "| Test:", len(test_ds))
xb, yb = next(iter(train_loader_plain))
print("One batch:", tuple(xb.shape), "= (batch, channels, height, width)")

In [ ]:
# What augmentation does: the same shirt, shown 8 times with random flips and shifts
fig, axes = plt.subplots(2, 8, figsize=(15, 4.4))
for ax in axes[0]:
    ax.imshow(raw_train.data[0].numpy(), cmap="gray")
    ax.set_title("Original", fontsize=9)
    ax.axis("off")
for ax in axes[1]:
    tensor, _ = train_aug[0]
    ax.imshow((tensor[0] * STD + MEAN).numpy(), cmap="gray")
    ax.set_title("Augmented", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Training image 0 ({CLASS_NAMES[int(raw_train.targets[0])]}): original and random augmentations")
plt.tight_layout()
plt.savefig("augmentation_examples.png", dpi=150)
plt.show()

## Step 3: Define the three models

1. **Fully connected baseline:** flatten the image, then two hidden layers.
2. **Small CNN:** two convolution blocks (16 and 32 filters), then a small classifier.
3. **Final CNN (`FashionNet`):** three convolution blocks (32, 64 and 128 filters). Each block is
   `Conv2d` + `BatchNorm2d` + `ReLU` + `MaxPool`.
   - **Batch normalisation** keeps the numbers inside the network well scaled, so training is faster and more stable.
   - **Dropout (0.4)** randomly switches off neurons while training, which reduces overfitting.
   - Image size goes 28 x 28 -> 14 x 14 -> 7 x 7 -> 3 x 3, while the number of feature maps grows 1 -> 32 -> 64 -> 128.

In [ ]:
def make_mlp():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256), nn.ReLU(),
        nn.Linear(256, 128), nn.ReLU(),
        nn.Linear(128, 10),
    )


def make_small_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
        nn.Linear(64, 10),
    )


class FashionNet(nn.Module):
    """Final CNN: three convolution blocks with batch normalisation, then a classifier with dropout."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),     # 14 x 14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),    # 7 x 7
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),  # 3 x 3
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters())


torch.manual_seed(42)
mlp = make_mlp()
torch.manual_seed(42)
small_cnn = make_small_cnn()
torch.manual_seed(42)
final_cnn = FashionNet()

print("Shape after each stage of the final CNN (one image):")
x = torch.zeros(1, 1, 28, 28)
final_cnn.eval()
for layer in final_cnn.features:
    x = layer(x)
    if layer.__class__.__name__ == "MaxPool2d":
        print(f"  after pooling -> {tuple(x.shape)}")
print(f"  classifier output -> {tuple(final_cnn.classifier(x).shape)}  (one score per category)")

params_mlp, params_small, params_final = count_params(mlp), count_params(small_cnn), count_params(final_cnn)
print(f"\nParameters: fully connected {params_mlp:,} | small CNN {params_small:,} | final CNN {params_final:,}")

## Step 4: Train the three models

Same optimizer (Adam, learning rate 0.001) and loss (`CrossEntropyLoss`) for all three.
After every epoch we measure the validation accuracy and keep the weights from the **best** epoch.
The final CNN trains on the augmented images; the two baselines train on the plain images.

In [ ]:
loss_fn = nn.CrossEntropyLoss()


def run_epoch(model, loader, optimizer=None):
    """One pass over the data. Trains if an optimizer is given, otherwise only measures."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            out = model(xb)                       # forward pass
            loss = loss_fn(out, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()                   # backpropagation
                optimizer.step()                  # update the weights
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(dim=1) == yb).sum().item()
            count += xb.size(0)
    return total_loss / count, correct / count


def train_model(model, name, loader, epochs, lr=LEARNING_RATE):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc, best_state = 0.0, None
    start = time.time()
    print(f"--- Training: {name} ---")
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        for key, value in zip(history, (tr_loss, va_loss, tr_acc, va_acc)):
            history[key].append(value)
        note = ""
        if va_acc > best_val_acc:
            best_val_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
            note = "  <- best so far"
        print(f"Epoch {epoch:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f}{note}")
    seconds = time.time() - start
    model.load_state_dict(best_state)
    print(f"Restored the best weights (validation accuracy {best_val_acc:.4f}) | training time {seconds:.0f} s\n")
    return history, best_val_acc, seconds

In [ ]:
mlp_hist, mlp_best, mlp_time = train_model(mlp, "1. Fully connected baseline", train_loader_plain, EPOCHS_MLP)
small_hist, small_best, small_time = train_model(small_cnn, "2. Small CNN", train_loader_plain, EPOCHS_SMALL)
final_hist, final_best, final_time = train_model(final_cnn, "3. Final CNN (batch norm + dropout + augmentation)",
                                                 train_loader_aug, EPOCHS_FINAL)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for hist, label, color in [(mlp_hist, "Fully connected", "tab:orange"),
                           (small_hist, "Small CNN", "tab:green"),
                           (final_hist, "Final CNN", "tab:blue")]:
    axes[0].plot(range(1, len(hist["val_acc"]) + 1), [v * 100 for v in hist["val_acc"]], marker="o", color=color, label=label)
axes[0].set_title("Validation accuracy of the three models")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)")

epochs_final = range(1, len(final_hist["train_loss"]) + 1)
axes[1].plot(epochs_final, final_hist["train_loss"], "--", color="tab:blue", label="Train (augmented)")
axes[1].plot(epochs_final, final_hist["val_loss"], "-", color="tab:blue", label="Validation")
axes[1].set_title("Final CNN: loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Cross-entropy loss")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## Step 5: Comparison table (test set)

Now the 10,000 test images are used, once, for all three models.

In [ ]:
def predict_test(model):
    """Probabilities for all 10,000 test images (in the same order as test_raw)."""
    model.eval()
    all_probs, all_true = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            all_probs.append(torch.softmax(model(xb), dim=1).numpy())
            all_true.append(yb.numpy())
    return np.concatenate(all_probs), np.concatenate(all_true)


probs_mlp, y_true = predict_test(mlp)
probs_small, _ = predict_test(small_cnn)
probs_final, _ = predict_test(final_cnn)

acc_mlp = float((probs_mlp.argmax(axis=1) == y_true).mean())
acc_small = float((probs_small.argmax(axis=1) == y_true).mean())
acc_final = float((probs_final.argmax(axis=1) == y_true).mean())

In [ ]:
models_table = [
    # name, parameters, augmentation, epochs, best validation accuracy, test accuracy, training seconds
    ("Fully connected", params_mlp, "No", EPOCHS_MLP, mlp_best, acc_mlp, mlp_time),
    ("Small CNN", params_small, "No", EPOCHS_SMALL, small_best, acc_small, small_time),
    ("Final CNN", params_final, "Yes", EPOCHS_FINAL, final_best, acc_final, final_time),
]

print(f"{'Model':<17}{'Parameters':>11}{'Augment':>9}{'Epochs':>8}{'Val acc':>9}{'Test acc':>10}{'Wrong':>7}{'Time':>8}")
print("-" * 79)
for name, params, aug, ep, val_acc, test_acc, secs in models_table:
    wrong_n = int(round((1 - test_acc) * len(y_true)))
    print(f"{name:<17}{params:>11,}{aug:>9}{ep:>8}{val_acc * 100:>8.2f}%{test_acc * 100:>9.2f}%{wrong_n:>7}{secs:>6.0f} s")

best_name = max(models_table, key=lambda row: row[4])[0]      # chosen by VALIDATION accuracy, not by test accuracy
print(f"\nBest model by validation accuracy: {best_name}")

table_md = ["| Model | Parameters | Augmentation | Epochs | Validation accuracy | Test accuracy | Wrong (of 10,000) | Training time |",
            "|---|---|---|---|---|---|---|---|"]
for name, params, aug, ep, val_acc, test_acc, secs in models_table:
    wrong_n = int(round((1 - test_acc) * len(y_true)))
    table_md.append(f"| {name} | {params:,} | {aug} | {ep} | {val_acc * 100:.2f}% | {test_acc * 100:.2f}% | {wrong_n} | {secs:.0f} s |")
Path("comparison_table.md").write_text("\n".join(table_md) + "\n", encoding="utf-8")
print("Saved comparison_table.md")

## Step 6: Per-category accuracy and confusion matrix (final CNN)

**Recall** for a category is the share of its test images that the model identified correctly.
In the confusion matrix, rows are the true category, columns are the predicted category, and every number off the diagonal is a mistake.

In [ ]:
y_pred = probs_final.argmax(axis=1)
confidence = probs_final.max(axis=1)


def confusion_matrix(y_true, y_pred, n_classes=10):
    """Rows = true category, columns = predicted category."""
    cm = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm


def top_confusions(cm, k=5):
    """The k most common mistakes as (count, true category, predicted category)."""
    off = cm.copy()
    np.fill_diagonal(off, 0)
    pairs = [(int(off[i, j]), i, j) for i in range(off.shape[0]) for j in range(off.shape[1]) if off[i, j] > 0]
    return sorted(pairs, reverse=True)[:k]


cm = confusion_matrix(y_true, y_pred)
recall = np.diag(cm) / cm.sum(axis=1)
precision = np.diag(cm) / np.maximum(cm.sum(axis=0), 1)

print(f"Final CNN test accuracy: {acc_final * 100:.2f}%\n")
print(f"{'Category':<14}{'Precision':>10}{'Recall':>9}")
print("-" * 33)
for c, name in enumerate(CLASS_NAMES):
    print(f"{name:<14}{precision[c] * 100:>9.1f}%{recall[c] * 100:>8.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1, 1.1]})
order_c = np.argsort(-recall)            # highest first, so the lowest ends up at the top of the chart
axes[0].barh([CLASS_NAMES[c] for c in order_c], recall[order_c] * 100, color="tab:blue")
axes[0].set_xlim(max(0, recall.min() * 100 - 10), 100)
axes[0].set_xlabel("Recall (%)")
axes[0].set_title("Accuracy by category (lowest at the top)")
axes[0].grid(axis="x", alpha=0.3)

axes[1].imshow(cm, cmap="Blues")
axes[1].set_title(f"Confusion matrix (test accuracy {acc_final * 100:.2f}%)")
axes[1].set_xlabel("Predicted category")
axes[1].set_ylabel("True category")
axes[1].set_xticks(range(10))
axes[1].set_yticks(range(10))
axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
axes[1].set_yticklabels(CLASS_NAMES)
for i in range(10):
    for j in range(10):
        axes[1].text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8,
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print("\nMost common mistakes (true -> predicted):")
for count, i, j in top_confusions(cm, 5):
    print(f"  {CLASS_NAMES[i]} -> {CLASS_NAMES[j]}: {count} times")

## Step 7: Prediction gallery

**Gallery 1:** 24 random test images. The title shows the predicted category and the confidence. **Green** = correct, **red** = wrong.

In [ ]:
rng = np.random.default_rng(0)
pick = rng.choice(len(y_true), size=24, replace=False)

fig, axes = plt.subplots(3, 8, figsize=(16, 7))
for ax, i in zip(axes.ravel(), pick):
    ax.imshow(test_raw[i], cmap="gray")
    ok = y_pred[i] == y_true[i]
    ax.set_title(f"{CLASS_NAMES[y_pred[i]]} ({confidence[i] * 100:.0f}%)" + ("" if ok else f"\nTrue: {CLASS_NAMES[y_true[i]]}"),
                 color="green" if ok else "red", fontsize=8)
    ax.axis("off")
plt.suptitle("Prediction gallery: 24 random test images (green = correct, red = wrong)")
plt.tight_layout()
plt.savefig("prediction_gallery.png", dpi=150)
plt.show()

## Step 8: One gallery per clothing category

Each row is one true category (shirts, trousers, bags, footwear and so on): the first 6 test images of that category,
with the model's prediction and confidence. This shows how reliable the model is for every kind of item.

In [ ]:
fig, axes = plt.subplots(10, 6, figsize=(10, 18))
for row in range(10):
    idx = np.where(y_true == row)[0][:6]
    for col in range(6):
        ax = axes[row, col]
        ax.axis("off")
        if col >= len(idx):
            continue
        i = idx[col]
        ax.imshow(test_raw[i], cmap="gray")
        ok = y_pred[i] == y_true[i]
        ax.set_title(f"{CLASS_NAMES[y_pred[i]]}\n{confidence[i] * 100:.0f}%", fontsize=7, color="green" if ok else "red")
    axes[row, 0].text(-0.15, 0.5, CLASS_NAMES[row], transform=axes[row, 0].transAxes,
                      ha="right", va="center", fontsize=10, fontweight="bold")
fig.subplots_adjust(left=0.2, right=0.98, top=0.93, bottom=0.01, hspace=0.75)
plt.suptitle("Predictions for each true category (green = correct, red = wrong)", y=0.985)
plt.savefig("category_gallery.png", dpi=150)
plt.show()

## Step 9: Look at the mistakes

The 10 mistakes the final CNN was **most confident** about.

In [ ]:
wrong = np.where(y_pred != y_true)[0]
print(f"The final CNN got {len(wrong)} of {len(y_true)} test images wrong.")
print(f"Average confidence: {confidence[y_pred == y_true].mean() * 100:.1f}% when correct, "
      f"{confidence[wrong].mean() * 100:.1f}% when wrong.\n")

worst = wrong[np.argsort(-confidence[wrong])][:10]
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for ax, i in zip(axes.ravel(), worst):
    ax.imshow(test_raw[i], cmap="gray")
    ax.set_title(f"True: {CLASS_NAMES[y_true[i]]}\nPredicted: {CLASS_NAMES[y_pred[i]]} ({confidence[i] * 100:.0f}%)",
                 color="red", fontsize=9)
    ax.axis("off")
plt.suptitle("The 10 most confident mistakes")
plt.tight_layout()
plt.savefig("misclassified.png", dpi=150)
plt.show()

## Step 10: Save the model and check that it reloads correctly

We save the final CNN's learned weights (`fashion_cnn.pth`) together with the category names and the normalisation numbers
(`model_info.json`). Then we build a fresh model, load the file and confirm it gives exactly the same test accuracy.

In [ ]:
MODEL_FILE = "fashion_cnn.pth"
torch.save(final_cnn.state_dict(), MODEL_FILE)
Path("model_info.json").write_text(json.dumps({
    "class_names": CLASS_NAMES,
    "mean": MEAN,
    "std": STD,
    "input_size": [1, 28, 28],
    "test_accuracy": round(acc_final, 4),
}, indent=2), encoding="utf-8")
print(f"Saved {MODEL_FILE} ({os.path.getsize(MODEL_FILE) / 1024:.0f} KB) and model_info.json")

loaded = FashionNet()
loaded.load_state_dict(torch.load(MODEL_FILE, map_location="cpu"))
probs_loaded, _ = predict_test(loaded)
acc_loaded = float((probs_loaded.argmax(axis=1) == y_true).mean())
if acc_loaded == acc_final:
    print(f"Reloaded model test accuracy: {acc_loaded * 100:.2f}% (identical, so the saved file works)")
else:
    print(f"WARNING: reloaded accuracy {acc_loaded * 100:.2f}% differs from {acc_final * 100:.2f}%. Run Steps 3 to 10 again.")

## Step 11 (optional): Try your own clothing photo

Put the path of a photo in `PHOTO_PATH`, for example `r"C:\Users\DELL\Pictures\shirt.jpg"`, and run the cell.
For the best result use a photo of **one item**, centred, on a **plain background**. The cell converts it to 28 x 28 grayscale
and flips it to light-on-dark if the background is bright, because Fashion-MNIST items are light on a dark background.
Real photos look different from the dataset, so expect some mistakes.

In [ ]:
PHOTO_PATH = ""      # example: r"C:\Users\DELL\Pictures\shirt.jpg"


def prepare_photo(path):
    img = Image.open(path).convert("L")
    corner = int(np.asarray(img)[0, 0])
    img = ImageOps.pad(img, (28, 28), color=corner)                 # square, keeping the proportions
    arr = np.asarray(img, dtype=np.float32) / 255.0
    if arr.mean() > 0.5:                                            # bright background -> invert
        arr = 1.0 - arr
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
    return (tensor - MEAN) / STD, arr


def predict_photo(path, model):
    tensor, arr = prepare_photo(path)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].numpy()
    top3 = probs.argsort()[::-1][:3]
    plt.figure(figsize=(3, 3))
    plt.imshow(arr, cmap="gray")
    plt.title(f"{CLASS_NAMES[top3[0]]} ({probs[top3[0]] * 100:.0f}%)")
    plt.axis("off")
    plt.show()
    for k in top3:
        print(f"  {CLASS_NAMES[k]:<12} {probs[k] * 100:5.1f}%")


if PHOTO_PATH and os.path.exists(PHOTO_PATH):
    predict_photo(PHOTO_PATH, loaded)
else:
    print("No photo chosen. Set PHOTO_PATH above to try your own picture (optional).")

## Step 12: Summary and conclusion

The cell below collects the key numbers and saves them as `results.json` and `results_summary.md` (ready to paste into your README).

In [ ]:
hardest = CLASS_NAMES[int(np.argmin(recall))]
easiest = CLASS_NAMES[int(np.argmax(recall))]
top_pair = top_confusions(cm, 1)[0]
gain_vs_mlp = (acc_final - acc_mlp) * 100
gain_vs_small = (acc_final - acc_small) * 100

summary = f"""## Results

{chr(10).join(table_md)}

- The final CNN reached **{acc_final * 100:.2f}%** test accuracy ({int(round((1 - acc_final) * len(y_true)))} wrong out of {len(y_true)}).
  That is {gain_vs_mlp:+.2f} points against the fully connected baseline and {gain_vs_small:+.2f} points against the small CNN.
- Easiest category: {easiest} ({recall.max() * 100:.1f}%). Hardest: {hardest} ({recall.min() * 100:.1f}%).
- Most common mistake: {CLASS_NAMES[top_pair[1]]} predicted as {CLASS_NAMES[top_pair[2]]} ({top_pair[0]} times).
- Saved model: `fashion_cnn.pth` (reloaded and checked: {'identical accuracy' if acc_loaded == acc_final else 'accuracy differs, rerun'}).
"""
print(summary)

Path("results_summary.md").write_text(summary, encoding="utf-8")
Path("results.json").write_text(json.dumps({
    "fully_connected": {"parameters": params_mlp, "val_acc": round(mlp_best, 4), "test_acc": round(acc_mlp, 4)},
    "small_cnn": {"parameters": params_small, "val_acc": round(small_best, 4), "test_acc": round(acc_small, 4)},
    "final_cnn": {"parameters": params_final, "val_acc": round(final_best, 4), "test_acc": round(acc_final, 4)},
    "hardest_category": hardest,
    "easiest_category": easiest,
}, indent=2), encoding="utf-8")

**Your conclusion** (4 to 6 sentences, using the numbers printed above):

- Which model was best, and which changes helped most (convolution, batch normalisation, dropout, augmentation)?
- Which clothing categories were easy and which were hard? Why?
- What are the limits of this model? (Think about tiny 28 x 28 grayscale images, and about real photos with colour and busy backgrounds.)
- What would you try next?

Your conclusion: